In [1]:
import torch
import torch.nn.functional as F
import time

def naive_attention(q, k, v):
    d = q.shape[-1]
    scores = q @ k.transpose(-2, -1)
    scores = scores / (d ** 0.5)
    weights = torch.softmax(scores, dim=-1)
    out = weights @ v
    return out

In [2]:
B = 1
H = 4
T = 128
D = 64

q = torch.randn(B, H, T, D, device="cuda", dtype=torch.float16)
k = torch.randn(B, H, T, D, device="cuda", dtype=torch.float16)
v = torch.randn(B, H, T, D, device="cuda", dtype=torch.float16)

out = naive_attention(q, k, v)
print(out.shape)

torch.Size([1, 4, 128, 64])


In [3]:
out_sdpa = F.scaled_dot_product_attention(q, k, v)

print(out_sdpa.shape)
print(torch.allclose(out, out_sdpa, atol=1e-2, rtol=1e-2))

torch.Size([1, 4, 128, 64])
True


In [4]:
def time_fn(fn, *args, repeat=20):
    torch.cuda.synchronize()
    t0 = time.perf_counter()

    for _ in range(repeat):
        y = fn(*args)

    torch.cuda.synchronize()
    return (time.perf_counter() - t0) / repeat


naive_time = time_fn(naive_attention, q, k, v)
sdpa_time = time_fn(F.scaled_dot_product_attention, q, k, v)

print("naive attention time:", naive_time)
print("PyTorch SDPA time:", sdpa_time)

naive attention time: 0.00014201275043888018
PyTorch SDPA time: 2.6845998945645988e-05


In [6]:
for T in [128, 512, 1024, 2048]:
    q = torch.randn(B, H, T, D, device="cuda", dtype=torch.float16)
    k = torch.randn(B, H, T, D, device="cuda", dtype=torch.float16)
    v = torch.randn(B, H, T, D, device="cuda", dtype=torch.float16)

    naive_time = time_fn(naive_attention, q, k, v, repeat=5)
    sdpa_time = time_fn(F.scaled_dot_product_attention, q, k, v, repeat=5)

    print(f"T={T} | naive={naive_time:.6f}s | sdpa={sdpa_time:.6f}s")

T=128 | naive=0.000211s | sdpa=0.000063s
T=512 | naive=0.000512s | sdpa=0.000147s
T=1024 | naive=0.000279s | sdpa=0.000076s
T=2048 | naive=0.000303s | sdpa=0.000094s


In [7]:
with torch.backends.cuda.sdp_kernel(
    enable_flash=True,
    enable_math=False,
    enable_mem_efficient=False,
):
    out_flash = F.scaled_dot_product_attention(q, k, v)

print(out_flash.shape)

torch.Size([1, 4, 2048, 64])


/home/mv/miniconda3/envs/env/lib/python3.10/contextlib.py:103: FutureWarning: `torch.backends.cuda.sdp_kernel()` is deprecated. In the future, this context manager will be removed. Please see `torch.nn.attention.sdpa_kernel()` for the new context manager, with updated signature.
  self.gen = func(*args, **kwds)
